# Jane Street Market Forecasting - PyTorch LSTM Baseline (Offline, Comparable)

This notebook trains a compact LSTM baseline for `responder_6` while keeping the validation contract comparable to `train_lgbm_baseline.ipynb`:

- Same Polars-first data loading and memory-aware downcasting
- Same base and engineered feature set
- Same chronological fold strategy and date windows
- Same train-only median imputation per fold
- Same competition metric: **sample-weighted zero-mean R^2**

The model differs by using recent same-symbol sequences instead of independent tabular rows. Validation labels remain restricted to the same validation dates as the LightGBM baseline.

In [ ]:
# Uncomment and run once if needed.
# !pip install torch polars pyarrow pandas scikit-learn

import copy
import gc
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [ ]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "jane-street-market-forecasting":
    PROJECT_DIR = PROJECT_DIR / "jane-street-market-forecasting"

KAGGLE_DATA_DIR = Path("/kaggle/input/competitions/jane-street-real-time-market-data-forecasting")
IS_KAGGLE = KAGGLE_DATA_DIR.exists()

DATA_DIR = KAGGLE_DATA_DIR if IS_KAGGLE else (PROJECT_DIR / "data")
TRAIN_DIR = DATA_DIR / "train.parquet"
TRAIN_GLOB = str(TRAIN_DIR / "partition_id=*" / "*.parquet")

TARGET = "responder_6"
WEIGHT_COL = "weight"
FEATURE_COLS = [f"feature_{i:02d}" for i in range(79)]
RESPONDER_COLS = [f"responder_{i}" for i in range(9)]
BASE_COLS = ["date_id", "time_id", "symbol_id", WEIGHT_COL] + FEATURE_COLS + RESPONDER_COLS

N_FOLDS = 1
DATE_GAP = 1
VAL_DAYS = 120  # roughly 6 months by date_id
TRAIN_LOOKBACK_DAYS = 360  # cap train window for memory; set None for full expanding window

# Optional cap for quick experiments / memory safety.
# Keep >= (N_FOLDS * VAL_DAYS + 180) for meaningful folds.
MAX_DATES = 500

# Match the LGBM baseline feature switches.
PHASE1_ENABLE = False
PHASE1_TOPK_BASE_FEATURES = 10
PHASE2_ENABLE = True
PHASE2_TOPK_BASE_FEATURES = 8
PHASE2_LAGS = [1, 2]
PHASE3_ENABLE = True
PHASE3_USE_RESPONDER_LAGS = True

# LSTM/GRU starting point. Tune gradually based on validation R^2.
SEQ_LEN = 16
BATCH_SIZE = 4096 if torch.cuda.is_available() else 1024
HIDDEN_SIZE = 128
NUM_LAYERS = 1
DROPOUT = 0.10
USE_GRU = False  # set True to train a GRU instead of an LSTM
STANDARDIZE_INPUTS = True  # per-feature z-score using train-fold stats (recommended for RNNs)
LR = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 10
PATIENCE = 3
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

# Set True for a very quick shape/training smoke test. Keep False for comparable runs.
FAST_DEV_RUN = False
FAST_DEV_MAX_TRAIN_SEQUENCES = 200_000
FAST_DEV_MAX_VAL_SEQUENCES = 50_000

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)

print(f"Running on Kaggle: {IS_KAGGLE}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"DATA_DIR exists: {DATA_DIR.exists()}")
print(f"TRAIN_DIR exists: {TRAIN_DIR.exists()}")
print(f"TRAIN_LOOKBACK_DAYS: {TRAIN_LOOKBACK_DAYS}")
print(f"MAX_DATES: {MAX_DATES}")
print(f"SEQ_LEN: {SEQ_LEN}")
print(f"BATCH_SIZE: {BATCH_SIZE}")
print(f"DEVICE: {DEVICE}")
print(f"FAST_DEV_RUN: {FAST_DEV_RUN}")

In [ ]:
def build_scan() -> pl.LazyFrame:
    scan = pl.scan_parquet(TRAIN_GLOB).select(BASE_COLS)

    if MAX_DATES is not None:
        max_date = scan.select(pl.max("date_id").alias("max_date")).collect().item()
        min_date = max_date - MAX_DATES + 1
        scan = scan.filter(pl.col("date_id") >= min_date)

    # Downcast to reduce memory pressure while preserving practical precision.
    cast_exprs = [
        pl.col("date_id").cast(pl.Int16),
        pl.col("time_id").cast(pl.Int16),
        pl.col("symbol_id").cast(pl.Int16),
        pl.col(WEIGHT_COL).cast(pl.Float32),
    ] + [pl.col(c).cast(pl.Float32) for c in FEATURE_COLS + RESPONDER_COLS]

    scan = scan.with_columns(cast_exprs)
    return scan


scan = build_scan()
df_pl = scan.collect(streaming=True).sort(["date_id", "time_id", "symbol_id"])

print(df_pl.shape)
print(df_pl.select(["date_id", "time_id", "symbol_id", WEIGHT_COL, TARGET]).head())
print(f"Estimated in-memory size (MB): {df_pl.estimated_size('mb'):.2f}")

In [ ]:
def add_basic_features_polars(frame: pl.DataFrame) -> pl.DataFrame:
    max_time = max(int(frame["time_id"].max()), 1)
    feature_exprs = [pl.col(c) for c in FEATURE_COLS]

    out = frame.with_columns(
        [
            pl.sum_horizontal([pl.col(c).is_null().cast(pl.Int16) for c in FEATURE_COLS])
            .cast(pl.Int16)
            .alias("feature_nan_count"),
            pl.mean_horizontal(feature_exprs).cast(pl.Float32).alias("feature_row_mean"),
            pl.mean_horizontal([pl.col(c).abs() for c in FEATURE_COLS])
            .cast(pl.Float32)
            .alias("feature_row_abs_mean"),
            ((2.0 * np.pi * pl.col("time_id").cast(pl.Float32)) / float(max_time))
            .sin()
            .cast(pl.Float32)
            .alias("time_sin"),
            ((2.0 * np.pi * pl.col("time_id").cast(pl.Float32)) / float(max_time))
            .cos()
            .cast(pl.Float32)
            .alias("time_cos"),
        ]
    )
    return out


def select_top_base_features_by_abs_corr(
    frame: pl.DataFrame,
    candidate_cols: list[str],
    target_col: str,
    top_k: int = 10,
) -> list[str]:
    corr_exprs = [pl.corr(pl.col(c), pl.col(target_col)).abs().alias(c) for c in candidate_cols]
    corr_row = frame.select(corr_exprs).row(0, named=True)

    sorted_feats = sorted(
        candidate_cols,
        key=lambda c: (corr_row[c] if corr_row[c] is not None else -1.0),
        reverse=True,
    )
    return sorted_feats[:top_k]


def add_cross_sectional_features_polars(
    frame: pl.DataFrame,
    selected_base_features: list[str],
) -> pl.DataFrame:
    group_cols = ["date_id", "time_id"]
    cs_exprs = []

    for c in selected_base_features:
        cs_exprs.append(
            (pl.col(c) - pl.col(c).mean().over(group_cols))
            .cast(pl.Float32)
            .alias(f"{c}_cs_demean")
        )
        cs_exprs.append(
            (
                pl.col(c).rank("average").over(group_cols).cast(pl.Float32)
                / pl.len().over(group_cols).cast(pl.Float32)
            )
            .cast(pl.Float32)
            .alias(f"{c}_cs_rank_pct")
        )

    return frame.with_columns(cs_exprs)


def add_prev_day_responder_lags_polars(
    frame: pl.DataFrame,
    responder_cols: list[str],
) -> pl.DataFrame:
    # Emulate lags.parquet availability: previous-date responder values by symbol.
    day_last_exprs = [
        pl.col(c).last().cast(pl.Float32).alias(f"{c}_day_last")
        for c in responder_cols
    ]

    daily = frame.group_by(["symbol_id", "date_id"]).agg(day_last_exprs).sort(["symbol_id", "date_id"])

    lag_cols = [f"lag1d_{c}" for c in responder_cols]
    lag_exprs = [
        pl.col(f"{c}_day_last")
        .shift(1)
        .over("symbol_id")
        .cast(pl.Float32)
        .alias(f"lag1d_{c}")
        for c in responder_cols
    ]

    lag_df = daily.with_columns(lag_exprs).select(["symbol_id", "date_id"] + lag_cols)

    out = frame.join(lag_df, on=["symbol_id", "date_id"], how="left")

    out = out.with_columns(
        [
            pl.mean_horizontal([pl.col(c) for c in lag_cols])
            .cast(pl.Float32)
            .alias("lag1d_resp_mean"),
            pl.mean_horizontal([pl.col(c).abs() for c in lag_cols])
            .cast(pl.Float32)
            .alias("lag1d_resp_abs_mean"),
            (
                pl.col("lag1d_responder_6")
                - pl.mean_horizontal([pl.col(c) for c in lag_cols])
            )
            .cast(pl.Float32)
            .alias("lag1d_target_vs_resp_mean"),
        ]
    )

    return out


def weighted_zero_mean_r2(y_true: np.ndarray, y_pred: np.ndarray, w: np.ndarray) -> float:
    num = np.sum(w * (y_true - y_pred) ** 2)
    den = np.sum(w * (y_true**2))
    if den == 0:
        return np.nan
    return 1.0 - num / den


def make_time_series_folds(
    unique_dates: np.ndarray,
    n_folds: int = 3,
    val_days: int = 180,
    gap: int = 1,
    train_lookback_days: int | None = 540,
):
    """
    Build chronological folds with fixed-size validation blocks near the dataset tail.
    Each fold validation span is `val_days` (about 6 months).
    """
    n_dates = len(unique_dates)
    required = n_folds * val_days + gap + 30
    if n_dates < required:
        raise ValueError(
            f"Not enough dates ({n_dates}) for {n_folds} folds with val_days={val_days}."
        )

    folds = []
    for k in range(n_folds):
        # Older fold first, newest fold last.
        val_end = n_dates - (n_folds - 1 - k) * val_days
        val_start = val_end - val_days
        train_end = val_start - gap

        if train_lookback_days is None:
            train_start = 0
        else:
            train_start = max(0, train_end - train_lookback_days)

        train_dates = unique_dates[train_start:train_end]
        val_dates = unique_dates[val_start:val_end]

        if len(train_dates) == 0 or len(val_dates) == 0:
            raise ValueError("Empty train/val split created. Increase MAX_DATES.")

        folds.append((train_dates, val_dates))

    return folds

In [ ]:
# Build baseline engineered dataset first.
if "df_pl" not in globals():
    scan = build_scan()
    df_pl = scan.collect(streaming=True).sort(["date_id", "time_id", "symbol_id"])

df_base_pl = add_basic_features_polars(df_pl)
del df_pl
gc.collect()

unique_dates = np.sort(df_base_pl["date_id"].unique().to_numpy())
folds = make_time_series_folds(
    unique_dates,
    n_folds=N_FOLDS,
    val_days=VAL_DAYS,
    gap=DATE_GAP,
    train_lookback_days=TRAIN_LOOKBACK_DAYS,
)

for i, (tr_dates, va_dates) in enumerate(folds, start=1):
    print(
        f"Fold {i}: train [{int(tr_dates.min())}, {int(tr_dates.max())}] ({len(tr_dates)} dates) | "
        f"val [{int(va_dates.min())}, {int(va_dates.max())}] ({len(va_dates)} dates, ~6 months)"
    )

if PHASE1_ENABLE:
    # Use first fold train window to pick a compact set of base features.
    phase1_train_dates = folds[0][0]
    phase1_train_pl = df_base_pl.filter(pl.col("date_id").is_in(phase1_train_dates))
    phase1_top_features = select_top_base_features_by_abs_corr(
        phase1_train_pl,
        FEATURE_COLS,
        TARGET,
        top_k=PHASE1_TOPK_BASE_FEATURES,
    )
    print(f"Phase 1 selected base features ({len(phase1_top_features)}): {phase1_top_features}")

    df_fe_pl = add_cross_sectional_features_polars(df_base_pl, phase1_top_features)
    del phase1_train_pl
else:
    phase1_top_features = []
    df_fe_pl = df_base_pl

if PHASE3_ENABLE:
    if PHASE3_USE_RESPONDER_LAGS:
        df_fe_pl = add_prev_day_responder_lags_polars(df_fe_pl, RESPONDER_COLS)

# Drop intermediate baseline frame once final feature frame is built.
if df_fe_pl is not df_base_pl:
    del df_base_pl
gc.collect()

MODEL_FEATURES = [
    c
    for c in df_fe_pl.columns
    if c not in {TARGET, WEIGHT_COL} and not c.startswith("responder_")
]

print(f"Model features after Phase 1+2+3: {len(MODEL_FEATURES)}")
print(f"Feature-engineered size (MB): {df_fe_pl.estimated_size('mb'):.2f}")

In [ ]:
def compute_train_medians(train_pl: pl.DataFrame, feature_cols: list[str]) -> dict[str, float]:
    medians = train_pl.select([pl.col(c).median().alias(c) for c in feature_cols]).row(0, named=True)
    clean_medians = {}
    for c, value in medians.items():
        if value is None or not np.isfinite(value):
            clean_medians[c] = 0.0
        else:
            clean_medians[c] = float(value)
    return clean_medians


def fill_features_with_medians(frame: pl.DataFrame, feature_cols: list[str], medians: dict[str, float]) -> pl.DataFrame:
    fill_exprs = [
        pl.col(c)
        .fill_null(medians[c])
        .fill_nan(medians[c])
        .cast(pl.Float32)
        .alias(c)
        for c in feature_cols
    ]
    return frame.with_columns(fill_exprs)


def make_sequence_endpoints(
    symbol_ids: np.ndarray,
    date_ids: np.ndarray,
    target_dates: np.ndarray,
    seq_len: int,
) -> np.ndarray:
    """Return row indices where a same-symbol sequence can end and be scored."""
    if len(symbol_ids) < seq_len:
        return np.empty(0, dtype=np.int64)

    endpoints = np.arange(seq_len - 1, len(symbol_ids), dtype=np.int64)
    same_symbol = symbol_ids[endpoints] == symbol_ids[endpoints - seq_len + 1]
    in_target_dates = np.isin(date_ids[endpoints], target_dates)
    return endpoints[same_symbol & in_target_dates]


class JaneStreetSequenceDataset(Dataset):
    def __init__(
        self,
        X: np.ndarray,
        y: np.ndarray,
        w: np.ndarray,
        endpoints: np.ndarray,
        seq_len: int,
    ) -> None:
        self.X = X
        self.y = y
        self.w = w
        self.endpoints = endpoints.astype(np.int64, copy=False)
        self.seq_len = seq_len

    def __len__(self) -> int:
        return len(self.endpoints)

    def __getitem__(self, idx: int):
        end = int(self.endpoints[idx])
        start = end - self.seq_len + 1
        x = torch.from_numpy(self.X[start : end + 1])
        y = torch.tensor(self.y[end], dtype=torch.float32)
        w = torch.tensor(self.w[end], dtype=torch.float32)
        return x, y, w


def prepare_fold_arrays(
    frame: pl.DataFrame,
    feature_cols: list[str],
    target_dates: np.ndarray,
    seq_len: int,
):
    sort_cols = ["symbol_id", "date_id", "time_id"]
    frame = frame.sort(sort_cols)

    X = np.ascontiguousarray(frame.select(feature_cols).to_numpy(), dtype=np.float32)
    y = np.ascontiguousarray(frame[TARGET].to_numpy(), dtype=np.float32)
    w = np.ascontiguousarray(frame[WEIGHT_COL].to_numpy(), dtype=np.float32)
    symbol_ids = np.ascontiguousarray(frame["symbol_id"].to_numpy())
    date_ids = np.ascontiguousarray(frame["date_id"].to_numpy())

    endpoints = make_sequence_endpoints(symbol_ids, date_ids, target_dates, seq_len)
    return X, y, w, endpoints


def fit_standardizer(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Per-feature mean/std from the train fold. Guards against zero-variance columns."""
    mean = X.mean(axis=0, keepdims=True).astype(np.float32)
    std = X.std(axis=0, keepdims=True)
    std = np.where(std < 1e-8, 1.0, std).astype(np.float32)
    return mean, std


def apply_standardizer(X: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return np.ascontiguousarray((X - mean) / std, dtype=np.float32)


def maybe_limit_endpoints(endpoints: np.ndarray, max_sequences: int | None, seed: int) -> np.ndarray:
    if max_sequences is None or len(endpoints) <= max_sequences:
        return endpoints
    rng = np.random.default_rng(seed)
    selected = rng.choice(len(endpoints), size=max_sequences, replace=False)
    selected.sort()
    return endpoints[selected]


def make_loader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=(DEVICE == "cuda"),
        drop_last=False,
    )

In [ ]:
class SequenceRegressor(nn.Module):
    def __init__(
        self,
        n_features: int,
        hidden_size: int = 128,
        num_layers: int = 1,
        dropout: float = 0.10,
        use_gru: bool = False,
    ) -> None:
        super().__init__()
        rnn_dropout = dropout if num_layers > 1 else 0.0
        rnn_cls = nn.GRU if use_gru else nn.LSTM
        self.rnn = rnn_cls(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=rnn_dropout,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 64),
            nn.SiLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # nn.GRU returns (output, h_n); nn.LSTM returns (output, (h_n, c_n)).
        out, _ = self.rnn(x)
        last_hidden = out[:, -1, :]
        return self.head(last_hidden).squeeze(-1)


def weighted_mse_loss(pred: torch.Tensor, target: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:
    return (weight * (target - pred).pow(2)).sum() / weight.sum().clamp_min(1e-12)


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer | None = None,
) -> dict[str, float]:
    is_train = optimizer is not None
    model.train(is_train)

    total_loss_num = 0.0
    total_weight = 0.0
    total_r2_num = 0.0
    total_r2_den = 0.0

    for X_batch, y_batch, w_batch in loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)
        w_batch = w_batch.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            pred = model(X_batch)
            loss = weighted_mse_loss(pred, y_batch, w_batch)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        with torch.no_grad():
            err2 = (y_batch - pred).pow(2)
            total_loss_num += float((w_batch * err2).sum().detach().cpu())
            total_weight += float(w_batch.sum().detach().cpu())
            total_r2_num += float((w_batch * err2).sum().detach().cpu())
            total_r2_den += float((w_batch * y_batch.pow(2)).sum().detach().cpu())

    weighted_loss = np.nan if total_weight == 0 else total_loss_num / total_weight
    weighted_r2 = np.nan if total_r2_den == 0 else 1.0 - (total_r2_num / total_r2_den)
    return {
        "weighted_mse": weighted_loss,
        "weighted_zero_mean_r2": weighted_r2,
        "r2_num": total_r2_num,
        "r2_den": total_r2_den,
    }


def train_one_fold(
    train_dataset: Dataset,
    val_dataset: Dataset,
    n_features: int,
) -> tuple[nn.Module, dict[str, float], dict[str, float], int]:
    model = SequenceRegressor(
        n_features=n_features,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        use_gru=USE_GRU,
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    train_loader = make_loader(train_dataset, BATCH_SIZE, shuffle=True)
    val_loader = make_loader(val_dataset, BATCH_SIZE, shuffle=False)

    best_state = copy.deepcopy(model.state_dict())
    best_val_r2 = -np.inf
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_metrics = run_epoch(model, train_loader, optimizer=optimizer)
        val_metrics = run_epoch(model, val_loader, optimizer=None)

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_metrics['weighted_mse']:.6f} | "
            f"train_r2={train_metrics['weighted_zero_mean_r2']:.6f} | "
            f"val_loss={val_metrics['weighted_mse']:.6f} | "
            f"val_r2={val_metrics['weighted_zero_mean_r2']:.6f}"
        )

        if val_metrics["weighted_zero_mean_r2"] > best_val_r2:
            best_val_r2 = val_metrics["weighted_zero_mean_r2"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f"Early stopping at epoch {epoch}; best_epoch={best_epoch}.")
                break

        if FAST_DEV_RUN:
            print("FAST_DEV_RUN enabled; stopping after one epoch.")
            break

    model.load_state_dict(best_state)
    final_train_metrics = run_epoch(model, train_loader, optimizer=None)
    final_val_metrics = run_epoch(model, val_loader, optimizer=None)
    return model, final_train_metrics, final_val_metrics, best_epoch

In [ ]:
fold_metrics = []
global_val_num = 0.0
global_val_den = 0.0

for fold_idx, (train_dates, val_dates) in enumerate(folds, start=1):
    train_pl = df_fe_pl.filter(pl.col("date_id").is_in(train_dates))
    val_pl = df_fe_pl.filter(pl.col("date_id").is_in(val_dates))

    medians = compute_train_medians(train_pl, MODEL_FEATURES)
    train_pl = fill_features_with_medians(train_pl, MODEL_FEATURES, medians)
    val_pl = fill_features_with_medians(val_pl, MODEL_FEATURES, medians)

    print(
        f"Fold {fold_idx} rows | train={train_pl.height:,}, val={val_pl.height:,} | "
        f"train_mb~{train_pl.estimated_size('mb'):.1f}, val_mb~{val_pl.estimated_size('mb'):.1f}"
    )

    X_train, y_train, w_train, train_endpoints = prepare_fold_arrays(
        train_pl, MODEL_FEATURES, train_dates, SEQ_LEN
    )
    X_val, y_val, w_val, val_endpoints = prepare_fold_arrays(
        val_pl, MODEL_FEATURES, val_dates, SEQ_LEN
    )

    del train_pl, val_pl
    gc.collect()

    if STANDARDIZE_INPUTS:
        feat_mean, feat_std = fit_standardizer(X_train)
        X_train = apply_standardizer(X_train, feat_mean, feat_std)
        X_val = apply_standardizer(X_val, feat_mean, feat_std)

    if FAST_DEV_RUN:
        train_endpoints = maybe_limit_endpoints(
            train_endpoints, FAST_DEV_MAX_TRAIN_SEQUENCES, seed=SEED + fold_idx
        )
        val_endpoints = maybe_limit_endpoints(
            val_endpoints, FAST_DEV_MAX_VAL_SEQUENCES, seed=SEED + 10_000 + fold_idx
        )

    train_dataset = JaneStreetSequenceDataset(X_train, y_train, w_train, train_endpoints, SEQ_LEN)
    val_dataset = JaneStreetSequenceDataset(X_val, y_val, w_val, val_endpoints, SEQ_LEN)

    print(
        f"Fold {fold_idx} sequences | train={len(train_dataset):,}, val={len(val_dataset):,} | "
        f"n_features={len(MODEL_FEATURES)}"
    )

    if len(train_dataset) == 0 or len(val_dataset) == 0:
        raise ValueError(
            f"Fold {fold_idx} has no train or validation sequences. Lower SEQ_LEN or increase date windows."
        )

    model, train_metrics, val_metrics, best_epoch = train_one_fold(
        train_dataset,
        val_dataset,
        n_features=len(MODEL_FEATURES),
    )

    global_val_num += val_metrics["r2_num"]
    global_val_den += val_metrics["r2_den"]

    fold_metrics.append(
        {
            "fold": fold_idx,
            "best_epoch": best_epoch,
            "train_weighted_zero_mean_r2": train_metrics["weighted_zero_mean_r2"],
            "val_weighted_zero_mean_r2": val_metrics["weighted_zero_mean_r2"],
            "train_weighted_mse": train_metrics["weighted_mse"],
            "val_weighted_mse": val_metrics["weighted_mse"],
            "train_sequences": len(train_dataset),
            "val_sequences": len(val_dataset),
        }
    )

    print(
        f"Fold {fold_idx} | best_epoch={best_epoch} | "
        f"train_r2={train_metrics['weighted_zero_mean_r2']:.6f} | "
        f"val_r2={val_metrics['weighted_zero_mean_r2']:.6f}"
    )

    del (
        X_train,
        y_train,
        w_train,
        train_endpoints,
        X_val,
        y_val,
        w_val,
        val_endpoints,
        train_dataset,
        val_dataset,
        model,
    )
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

metrics_df = pd.DataFrame(fold_metrics)
metrics_df

In [ ]:
overall_oof_r2 = np.nan if global_val_den == 0 else 1.0 - (global_val_num / global_val_den)

print("\nTrain/Eval metrics summary")
print(metrics_df.to_string(index=False))
print(f"\nMean fold train weighted zero-mean R^2: {metrics_df['train_weighted_zero_mean_r2'].mean():.6f}")
print(f"Mean fold val weighted zero-mean R^2: {metrics_df['val_weighted_zero_mean_r2'].mean():.6f}")
print(f"Overall OOF weighted zero-mean R^2: {overall_oof_r2:.6f}")

## Gradual Tuning Notes

Start with the defaults above and compare `val_weighted_zero_mean_r2` plus `overall_oof_r2` to the LightGBM baseline.

Suggested tuning order:

1. Sequence length: `8`, `16`, `32`, then `64` only if training remains stable.
2. Hidden size: `64`, `128`, `256`.
3. Depth: try `NUM_LAYERS = 2` and keep dropout enabled inside the LSTM.
4. Learning rate: try `5e-4` and `2e-3` after picking a stable sequence length/capacity.

For a quick notebook smoke test, set `FAST_DEV_RUN = True`; for comparable runs, keep it `False` and leave the split, features, and metric unchanged.